### step 1 Data Integration

In [20]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [21]:
def process_all_pdf(pdf_dirs):
    all_docs=[]
    pdf_dir=Path(pdf_dirs)
    pdf_files=list(pdf_dir.glob("*.pdf"))
    print(len(pdf_files))
    for file in pdf_files:
        try:
            loader=PyMuPDFLoader(str(file))
            document=loader.load()
            for doc in document:
                doc.metadata['source_file']=file.name
                doc.metadata['file_type']='pdf'
                
            all_docs.extend(document)
        except Exception as e:
            print(e)
            
    return all_docs

all_pdf_docs=process_all_pdf("data/PDF")
all_pdf_docs

3


[Document(metadata={'producer': 'WeasyPrint 68.1', 'creator': '', 'creationdate': '', 'source': 'data\\PDF\\first.pdf', 'file_path': 'data\\PDF\\first.pdf', 'total_pages': 7, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'first.pdf', 'file_type': 'pdf'}, page_content='Price Per Token Guide to Using Fewer\nTokens\nMost developers use 2-5x more tokens than they need to. Here\'s how to fix that.\nThis guide covers eight practical ways to cut your token usage. Each one is independent\n— you can start with whichever applies to you — but they compound when used together.\nTip 1: Trim Your System Prompt\nYour system prompt gets sent with every single API call. There\'s no getting around that if\nyou\'re using an LLM as an assistant because each call is stateless- all the context needs\nto be included in every input. It is worth spending time on a frequently used system p

### step 2 Chunking

In [22]:
def split_docs(documents,chunk_size=1000,chunk_overlap=200):
    text_sp=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n','\n',' ','']
    )
    split_docs=text_sp.split_documents(documents)
    return split_docs

chunks=split_docs(documents=all_pdf_docs)
chunks

[Document(metadata={'producer': 'WeasyPrint 68.1', 'creator': '', 'creationdate': '', 'source': 'data\\PDF\\first.pdf', 'file_path': 'data\\PDF\\first.pdf', 'total_pages': 7, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'first.pdf', 'file_type': 'pdf'}, page_content='Price Per Token Guide to Using Fewer\nTokens\nMost developers use 2-5x more tokens than they need to. Here\'s how to fix that.\nThis guide covers eight practical ways to cut your token usage. Each one is independent\n— you can start with whichever applies to you — but they compound when used together.\nTip 1: Trim Your System Prompt\nYour system prompt gets sent with every single API call. There\'s no getting around that if\nyou\'re using an LLM as an assistant because each call is stateless- all the context needs\nto be included in every input. It is worth spending time on a frequently used system p

### step 3 Embedding


In [23]:
import uuid
import numpy as np
from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
import chromadb
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [25]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()
    def _load_model(self):
        try:
            self.model=SentenceTransformer(self.model_name)
        except Exception as e:
            print(e)
        
    
    def generate_embedding(self,texts:List[str])->np.ndarray:
        if not self.model:
            raise ValueError("model not loaded")
        emb=self.model.encode(texts,show_progress_bar=True)
        return emb
    
emb_manager = EmbeddingManager()
emb_manager

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3466.01it/s]


In [26]:
class VectorStore:
    def __init__(self,collection_name:str="pdf_documents",per_dir:str='data/vector_store'):
        self.collection_name=collection_name
        self.per_dir=per_dir
        self.client=None
        self.collection=None
        self._initialize_store()
    
    def _initialize_store(self):
        try:
            os.makedirs(self.per_dir,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.per_dir)
            
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={'description':'bla bla bla'}
            )
        except Exception:
            print(Exception)
            raise
    
    def add_elements(self,documents:List[Any],embeddings:np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("values mis match")
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]
        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
        except Exception as e:
            print(e)
            raise
        
        

In [27]:
vectorstore=VectorStore()
texts=[doc.page_content for doc in chunks]
embeddings=emb_manager.generate_embedding(texts)
vectorstore.add_elements(chunks,embeddings)

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.02it/s]


In [37]:
class RAGRetriever:
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager):
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager
        
    def retrieve(self, query: str,top_k:int=5, score_threshold:float=0.0)->List[Dict[str,Any]]:
        
        query_emb=self.embedding_manager.generate_embedding([query])[0]
        try:
            result=self.vector_store.collection.query(
                query_embeddings=[query_emb.tolist()],
                n_results=top_k
                
            )
            print("="*50)
            print(result)
            print("="*50)
            retrieved_docs=[]
            if result['documents'] and result['documents'][0]:
                documents=result['documents'][0]
                metadatas=result['metadatas'][0]
                distances=result['distances'][0]
                ids=result['ids'][0]
                
                for i,(doc_id,document,metadata,distance) in enumerate(
                    zip(ids,documents,metadatas,distances)
                ):
                    print(f"Distance: {distance}")

                    similarity_score = distance

                    print(f"Similarity: {similarity_score}")

                    retrieved_docs.append({
                        'id': doc_id,
                        'content': document,
                        'metadata': metadata,
                        'distance': distance,
                        'rank': i + 1
                    })
                print(len(retrieved_docs))
            else:
                print("no document found")
                
            return retrieved_docs          
        except Exception as e:
            print(e)
            return []
        
rag_retriever=RAGRetriever(vectorstore,emb_manager)
rag_retriever

In [38]:
rag_retriever.retrieve("Control Your Output Length")

Batches: 100%|██████████| 1/1 [00:00<00:00, 46.46it/s]

{'ids': [['doc_ee7ffa59_3', 'doc_537e4670_3', 'doc_f794c9f6_16', 'doc_948430ef_16', 'doc_8a352251_0']], 'embeddings': None, 'documents': [["say so honestly rather than making something up. Format your responses clearly\nwith bullet points where appropriate...\nAfter (roughly 200 tokens):\nCustomer support agent for [Company]. Look up order details when asked.\nBe concise. If unsure, say so. Use the order_lookup tool for order queries.\nSame behavior. 600 fewer tokens per call. At 200 calls/day, that's 120,000 tokens saved\ndaily — purely from deleting words the model didn't need. Details about looking up order\ndates, items ordered, etc can be included in tool definitions and picked up into context only\nwhen necessary.\nTip 2: Control Your Output Length\nIf you don't tell the model how long to respond, it will use as many tokens as it wants. This\nmatters because output tokens are the expensive ones — typically 3-5x the cost of input\ntokens.\nA few ways to control this:\nSet max_toke

[{'id': 'doc_ee7ffa59_3',
  'content': "say so honestly rather than making something up. Format your responses clearly\nwith bullet points where appropriate...\nAfter (roughly 200 tokens):\nCustomer support agent for [Company]. Look up order details when asked.\nBe concise. If unsure, say so. Use the order_lookup tool for order queries.\nSame behavior. 600 fewer tokens per call. At 200 calls/day, that's 120,000 tokens saved\ndaily — purely from deleting words the model didn't need. Details about looking up order\ndates, items ordered, etc can be included in tool definitions and picked up into context only\nwhen necessary.\nTip 2: Control Your Output Length\nIf you don't tell the model how long to respond, it will use as many tokens as it wants. This\nmatters because output tokens are the expensive ones — typically 3-5x the cost of input\ntokens.\nA few ways to control this:\nSet max_tokens  explicitly. Most people leave this at the default, which is often the",
  'metadata': {'source':